# Lecture 2: makemore — 字符级 Bigram 语言模型

本笔记本跟随 Andrej Karpathy 的 [Neural Networks: Zero to Hero](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) 系列课程第 2 讲，构建一个字符级的 bigram 语言模型来生成人名。

**学习路线：**
1. 数据加载与 bigram 统计（字典方法）
2. 用张量构建 27×27 计数矩阵并可视化
3. 将计数转为概率矩阵，采样生成名字
4. 用对数似然（log-likelihood）评估模型质量
5. 用神经网络（单层 Softmax）替代查表法
6. 梯度下降优化神经网络权重

## 1. 数据加载与探索

**概念解释：** 我们使用一个包含 32,000+ 人名的数据集 `names.txt`。每个名字是一个字符序列，我们的目标是学习字符之间的统计规律，从而生成新的、听起来合理的名字。

这是一个**生成式模型（Generative Model）**的入门案例：不是分类或预测，而是"创造"新数据。

In [ ]:
word=open('D:\\Vault-4\\Projects\\makemore\\names.txt','r').read().splitlines()  # 读取名字数据集，每行一个名字

In [2]:
word[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
len(word)


32033

In [4]:
min(len(w) for w in word)

2

In [5]:
max(len(w) for w in word)

15

## 2. Bigram 统计（字典方法）

**概念解释：** **Bigram** 是相邻的两个字符组成的对（pair）。例如 `emma` 产生的 bigram 有：`<S>e`, `em`, `mm`, `ma`, `a<E>`。

通过统计整个数据集中每个 bigram 出现的次数，我们就能知道"在某个字符后面，哪些字符更常出现"——这就是语言模型的本质。

**生活类比：** 想象你经常打字，输入法会提示"下一个字"——它就是基于类似的 bigram/n-gram 统计。

In [6]:
# b 是一个字典，用来统计每个 bigram（相邻字符对）出现的次数
# 键: (ch1, ch2) 元组，值: 出现次数
b={}

# 遍历前 2 个单词（word[:2] → ['emma', 'olivia']）
for w in word:

    # 构造带有【起始标记】和【终止标记】的字符列表
    # 例如 w = 'emma':
    #   list(w)  = ['e', 'm', 'm', 'a']
    #   chs      = ['<S>', 'e', 'm', 'm', 'a', '<E>']
    #
    # '<S>' 表示单词的开始（Start），'<E>' 表示单词的结束（End）
    # 这样我们就能学到：
    #   - '<S>' → 'e'：单词倾向以 'e' 开头
    #   - 'a' → '<E>'：单词倾向以 'a' 结尾
    chs=['<S>']+list(w)+['<E>'] 

    # 用 zip 滑动窗口提取所有相邻字符对（bigram）
    # chs      = ['<S>', 'e', 'm', 'm', 'a', '<E>']
    # chs[1:]  = ['e',   'm', 'm', 'a', '<E>']
    # 配对结果 → ('<S>','e'), ('e','m'), ('m','m'), ('m','a'), ('a','<E>')
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram=(ch1, ch2)

        # dict.get(key, default) 的用法：
        #   d[key]             → 找不到就报 KeyError（激进派）
        #   d.get(key, default)→ 找不到就返回 default（温和派）
        #   d.get(key)         → 找不到就返回 None
        #
        # 举例：basket = {'apple': 3, 'banana': 5}
        #   basket.get('apple', 0)  → 3   （存在，返回实际值）
        #   basket.get('orange', 0) → 0   （不存在，返回默认值 0）
        #
        # 本行的执行过程（以 bigram ('m','m') 为例）：
        #   第一次遇到: b.get(('m','m'), 0) → 0（不存在）, b[('m','m')] = 0+1 = 1
        #   第二次遇到: b.get(('m','m'), 0) → 1（已存在）, b[('m','m')] = 1+1 = 2
        #
        # 等价于:
        #   if bigram not in b:
        #       b[bigram] = 0
        #   b[bigram] = b[bigram] + 1
        b[bigram]=b.get(bigram, 0)+1

In [7]:
sorted(b.items(), key=lambda x: x[1], reverse=True)[:10]

[(('n', '<E>'), 6763),
 (('a', '<E>'), 6640),
 (('a', 'n'), 5438),
 (('<S>', 'a'), 4410),
 (('e', '<E>'), 3983),
 (('a', 'r'), 3264),
 (('e', 'l'), 3248),
 (('r', 'i'), 3033),
 (('n', 'a'), 2977),
 (('<S>', 'k'), 2963)]

## 3. 计数矩阵（张量方法）

**概念解释：** 用 Python 字典统计 bigram 虽然直观，但效率不高。我们改用一个 **27×27 的二维张量 `N`** 来存储所有 bigram 的计数。

- 行 = 第一个字符（共 27 种：`.` + `a-z`）
- 列 = 第二个字符
- `N[i][j]` = 字符 `itos[i]` 后面紧跟字符 `itos[j]` 的次数

$$N_{ij} = \text{count}(\text{char}_i \to \text{char}_j)$$

**生活类比：** 这就像一张"交叉频率表"——类似于调查"喜欢咖啡的人中有多少也喜欢茶"的统计表格。

In [8]:
import torch

In [ ]:
# N=torch.zeros((28,28), dtype=torch.int32)
N=torch.zeros((27,27), dtype=torch.int32)  # 27×27 计数矩阵：26 个字母 + 1 个特殊符号 '.'

In [ ]:
chars=sorted(list(set(''.join(word))))          # 提取所有出现过的字符并排序
stoi={s:i+1 for i,s in enumerate(chars)}        # 字符→索引映射，a=1, b=2, ..., z=26
stoi['.']=0                                     # 特殊符号 '.' 占索引 0（代替 <S> 和 <E>）
itos={i:s for s,i in stoi.items()}              # 索引→字符 的反向映射

In [11]:
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [ ]:
for w in word:                                    # 遍历所有名字
    chs=['.']+list(w)+['.']                       # 添加起止符号 '.'
    for ch1, ch2 in zip(chs, chs[1:]):            # 取相邻字符对（bigram）
        ix1=stoi[ch1]                             # 前一个字符的索引
        ix2=stoi[ch2]                             # 后一个字符的索引
        N[ix1, ix2]+=1                            # 在计数矩阵对应位置 +1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')                       # 用蓝色热力图显示计数矩阵
for i in range(N.shape[0]):
    for j in range(N.shape[1]):
        chstr=itos[i]+itos[j]                     # 该位置对应的 bigram 字符对
        plt.text(j, i, chstr, ha='center', va='bottom', color='gray')
        plt.text(j,i,N[i,j].item(), ha='center', va='top', color='gray')  # 显示计数值
    plt.axis('off')

## 4. 概率矩阵与采样

**概念解释：** 计数矩阵告诉我们"发生了多少次"，但要生成新名字，我们需要的是**概率**——"下一个字符是什么的可能性"。

将计数矩阵的每一行归一化（除以该行之和），就得到了概率矩阵 $P$：

$$P_{ij} = \frac{N_{ij}}{\sum_j N_{ij}}$$

每一行 $P_i$ 是一个概率分布（和为 1），表示"在字符 $i$ 之后，各字符出现的概率"。

然后用 `torch.multinomial` 按概率抽样，就能逐字符生成名字。

**生活类比：** 像扔一个不均匀的骰子——每面的概率不同，由训练数据决定。

In [14]:
N[0,:]
N[0]

tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929], dtype=torch.int32)

In [ ]:
p=N[0].float()    # 取第 0 行（'.' 后面跟什么字符），转为浮点数
p=p/p.sum()       # 归一化为概率分布（所有元素之和 = 1）
p

In [ ]:
P=N.float()
P=P/P.sum(1, keepdim=True)  # 按行归一化：每行除以该行之和；keepdim=True 保持维度以便广播
P

In [ ]:
g=torch.Generator().manual_seed(2147483647)       # 固定随机种子，保证可复现
ix=torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率 p 抽样一个索引
itos[ix]                                          # 将索引转回字符

In [ ]:
g=torch.Generator().manual_seed(2147483647)

for i in range(5):                                # 生成 5 个名字
    out=[]
    ix=0                                          # 从 '.'（起始符）开始
    while True:
        p=P[ix]                                   # 取当前字符对应的概率行

        # p=N[ix].float()
        # p=p/p.sum()

        # p=torch.ones(27)/27

        ix=torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率采样下一个字符
        out.append(itos[ix])
        # print(itos[ix], end='')
        if ix==0:                                 # 采到 '.'（终止符）则结束
            break

    print(''.join(out))

In [19]:
# # 抽取 100 次，统计每个字符被抽中的频率，与理论概率 p 对比
# from collections import Counter

# samples = [torch.multinomial(p, num_samples=1, replacement=True).item() for _ in range(100)]
# counts = Counter(samples)

# # 按频率从高到低排序，显示：字符 | 抽中次数 | 实际频率 | 理论概率
# print(f"{'字符':>4} {'次数':>4} {'实际频率':>8} {'理论概率':>8}")
# print("-" * 30)
# for ix, cnt in counts.most_common():
#     print(f"{itos[ix]:>4} {cnt:>4} {cnt/100:>8.2%} {p[ix].item():>8.2%}")

### 穿插练习：PyTorch 基础操作

以下几个 cell 是关于 `torch.multinomial`、张量创建与索引的基础练习，帮助熟悉 PyTorch 的核心 API。可跳过。

In [ ]:
g=torch.Generator().manual_seed(2147483647)
p=torch.rand(3, generator=g)   # 随机生成 3 个数
p=p/p.sum()                    # 归一化为概率分布
p
# sum(p)

In [21]:
torch.multinomial(p, num_samples=20, replacement=True, generator=g)

tensor([1, 1, 2, 0, 0, 2, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1])

In [22]:
a=torch.zeros((3,5), dtype=torch.int32)
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [23]:
a.dtype

torch.int32

In [24]:
a[1,3]+=1

In [25]:
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [26]:
a[0,0]=5

In [27]:
a

tensor([[5, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

## 5. 模型评估：对数似然（Log-Likelihood）

**概念解释：** 如何衡量模型的好坏？我们用**似然函数（Likelihood）**：给定训练数据，模型分配给它们的总概率越高，模型越好。

由于概率连乘容易下溢（变成极小的数），我们取对数将乘法变加法：

$$\log \mathcal{L} = \sum_{(x,y) \in \text{data}} \log P(y \mid x)$$

实际使用**负对数似然（Negative Log-Likelihood, NLL）** 作为损失函数——越小越好：

$$\text{NLL} = -\log \mathcal{L}$$

**生活类比：** 类似于考试评分——给模型看它认为"正确答案"的概率有多高。概率高 = 分数低（loss 小）= 模型好。

In [ ]:
log_likelihood=0.0                                # 累计对数似然
for w in word[:3]:                                # 取前 3 个单词
    chs=['.']+list(w)+['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]                          # 模型预测的 bigram 概率
        logprob=torch.log(prob)                   # 取对数
        log_likelihood+=logprob                   # 累加
        # N[ix1, ix2]+=1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll=-log_likelihood                               # 负对数似然（Negative Log-Likelihood）
print(f'negative log likelihood: {nll:.4f}')
anll=nll/len(word[:3])                            # 平均负对数似然 = 损失函数
print(f'average negative log likelihood: {anll:.4f}')

In [ ]:
log_likelihood=0.0
for w in ["qg"]:                                  # 测试罕见 bigram "qg"
    chs=['.']+list(w)+['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]
        logprob=torch.log(prob)
        log_likelihood+=logprob
        # N[ix1, ix2]+=1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll=-log_likelihood
print(f'negative log likelihood: {nll:.4f}')
anll=nll/len(word[:3])
print(f'average negative log likelihood: {anll:.4f}')

### 平滑处理（Smoothing）

**问题：** 如果某个 bigram 在训练数据中从未出现过（比如 `qg`），概率为 0，`log(0) = -∞`，损失爆炸！

**解决：** 给计数矩阵加一个常数（拉普拉斯平滑 / Add-k Smoothing）：

$$P_{ij} = \frac{N_{ij} + k}{\sum_j (N_{ij} + k)}$$

- $k=1$：轻微平滑，基本保持原始分布
- $k \to \infty$：概率趋向均匀分布（$1/27$），完全忽略数据

下面实验不同的 $k$ 值对罕见 bigram `"qg"` 损失的影响。

In [ ]:
# P=(N+1).float() # no zero
# P=(N+500).float()
# P=(N+1000).float()
P=(N+1000000).float()                             # 加极大平滑值 → 概率趋向均匀分布
# P=torch.ones_like(P) # 如果完全不使用 N 的统计结果，而是直接使用均匀分布（每个 bigram 出现的概率相同）
P=P/P.sum(1, keepdim=True) #这里需要注意广播机制：用矩阵除以矩阵，而不是用一维向量
# P

In [31]:
log_likelihood=0.0
for w in ["qg"]:
    chs=['.']+list(w)+['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]
        logprob=torch.log(prob)
        log_likelihood+=logprob
        # N[ix1, ix2]+=1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'log likelihood: {log_likelihood:.4f}')
nll=-log_likelihood
print(f'negative log likelihood: {nll:.4f}')
anll=nll/len(word[:3])
print(f'average negative log likelihood: {anll:.4f}')

log likelihood: -9.8886
negative log likelihood: 9.8886
average negative log likelihood: 3.2962


## 6. 神经网络方法：从查表到学习

**概念解释：** 前面的统计方法直接从数据"数"出概率，现在我们换一种方式：用**神经网络学习**这些概率。

核心思路（Softmax 回归）：
1. **输入：** 将字符索引转为 **one-hot 向量**（27 维，只有一位是 1）
2. **线性层：** 乘以权重矩阵 $W$，得到 **logits**（未归一化的分数）
3. **Softmax：** 对 logits 取 $\exp$ 再归一化，得到概率分布

$$\text{logits} = x_{\text{one-hot}} \cdot W$$
$$P(y \mid x) = \text{softmax}(\text{logits}) = \frac{e^{\text{logits}_y}}{\sum_j e^{\text{logits}_j}}$$

**生活类比：** 统计方法像"直接数投票"，神经网络方法像"训练一个裁判来预测投票结果"——两种方法在简单情况下结果等价，但神经网络能扩展到更复杂的模型。

- **One-hot 编码：** 把离散的字符变成数值向量，让矩阵乘法能处理
- **Logits → Softmax：** 将任意实数变成合法概率分布（非负、和为 1）

In [ ]:
# create the training set
xs,ys=[],[]                                       # xs: 输入字符索引，ys: 目标字符索引
for w in word[:1]:                                # 先用第 1 个单词 'emma' 测试
    chs=['.']+list(w)+['.']
    for ch1, ch2 in zip(chs, chs[1:]):            # 取 bigram 对
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        print(f'{ch1}{ch2}: {ix1} {ix2}')
        xs.append(ix1)                            # 输入：当前字符
        ys.append(ix2)                            # 标签：下一个字符

xs=torch.tensor(xs)                               # 转为 PyTorch 张量
ys=torch.tensor(ys)
xs, ys

In [33]:
import torch.nn.functional as F


In [ ]:
xenc=F.one_hot(xs, num_classes=27).float()  # 将输入索引转为 one-hot 向量（27 维）
xenc
xenc.shape                                  # (5, 27) — 5 个样本，每个 27 维
plt.imshow(xenc)                            # 可视化 one-hot 矩阵

In [ ]:
yenc=F.one_hot(ys, num_classes=27).float()  # 将标签也转为 one-hot（用于可视化对比）
yenc
yenc.shape
plt.imshow(yenc)

In [ ]:
# W=torch.randn((27,1))
W=torch.randn((27,2))     # 试验：27 维输入映射到 2 维输出
xenc @ W                   # 矩阵乘法：(5,27) @ (27,2) → (5,2)

In [37]:
# SUMMRY


In [38]:
xs

tensor([ 0,  5, 13, 13,  1])

In [39]:
ys

tensor([ 5, 13, 13,  1,  0])

### Softmax 前向传播与损失计算

下面我们用完整的 27×27 权重矩阵，跑一遍前向传播流程，并逐步计算每个 bigram 的负对数似然损失（NLL）。这有助于理解神经网络是如何"打分"的。

In [ ]:
g=torch.Generator().manual_seed(2147483647)
w=torch.rand((27,27), generator=g)  # 初始化 27×27 权重矩阵（输出也是 27 个字符的 logits）

In [ ]:
xenc=F.one_hot(xs, num_classes=27).float()  # one-hot 编码
logits=xenc @ w                             # 线性层：得到 logits（未归一化的对数概率）
counts=logits.exp()                         # exp 将 logits 转为正数（类似"计数"）
probs=counts/counts.sum(1, keepdim=True)    # 归一化为概率（这就是 Softmax！）

In [42]:
probs.shape

torch.Size([5, 27])

In [ ]:
nlls=torch.zeros(5) 
for i in range(5):                                    # 逐个 bigram 计算损失
    #i-th bigram: 
    x=xs[i].item()                                    # 输入字符索引
    y=ys[i].item()                                    # 标签字符索引
    print(f'bigram example {i+1}: ({itos[x]} -> {itos[y]}) (indexes: ({x}, {y}))')
    print('input to the neural net:', x)
    print('output probabilities from the neural net:', probs[i])
    print('label (actual next character):', y)
    p=probs[i, y]                                     # 模型给正确字符分配的概率
    print('probability assigned by the net to the correct character:', p.item())
    logp=torch.log(p)                                 # 取对数
    print('log likelihood:', logp.item())
    nll=-logp                                         # 负对数似然
    print('negative loglikelihood:', nll.item())
    nlls[i]=nll
    print('---')
    print('---')


print('average negative log likelihood:', nlls.mean().item())  # 平均 NLL = 损失

## 7. 梯度下降优化（Gradient Descent）

**概念解释：** 现在我们有了损失函数（NLL），如何让模型变好？答案是**梯度下降**：

1. **前向传播：** 计算预测概率和损失
2. **反向传播（Backpropagation）：** 计算损失对每个权重的梯度 $\frac{\partial L}{\partial W}$
3. **参数更新：** $W \leftarrow W - \eta \cdot \nabla_W L$（$\eta$ 是学习率）

重复以上步骤，损失逐渐下降，模型越来越好。

$$W_{\text{new}} = W_{\text{old}} - \eta \cdot \frac{\partial \text{Loss}}{\partial W}$$

**生活类比：** 像蒙眼下山——你看不到山的全貌，但能感受脚下的坡度（梯度），每一步沿着最陡的下坡方向走一小步，最终到达山谷（最低损失）。

- 学习率 $\eta$ 太大 → 步子太大，可能越过谷底
- 学习率 $\eta$ 太小 → 步子太小，收敛极慢

In [44]:
#   optimazation  #

In [148]:
xs

tensor([ 0,  5, 13, 13,  1])

In [149]:
ys

tensor([ 5, 13, 13,  1,  0])

In [ ]:
g=torch.Generator().manual_seed(2147483647)
W=torch.rand((27,27), generator=g, requires_grad=True)  # requires_grad=True 让 PyTorch 追踪梯度

In [ ]:
xenc=F.one_hot(xs, num_classes=27).float()
logits=xenc @ W                              # 前向传播：线性层
counts=logits.exp()                          # Softmax 第一步：取指数
probs=counts/counts.sum(1, keepdim=True)     # Softmax 第二步：归一化
loss=-probs[torch.arange(5),ys].log().mean() # 负对数似然损失（向量化写法）
loss

In [ ]:
W.grad=None                # 清零梯度（避免累加）
loss.backward()            # 反向传播：计算 dL/dW
W.data+=-0.5*W.grad        # 梯度下降：W = W - lr * grad（lr=0.5）

In [ ]:
LOSS=[]                                                # 记录每轮损失用于绘图
NUM_ITER=100
g=torch.Generator().manual_seed(2147483647)
W=torch.rand((27,27), generator=g, requires_grad=True)
for i in range(NUM_ITER):
    # --- 前向传播 ---
    xenc=F.one_hot(xs, num_classes=27).float()
    logits=xenc @ W
    counts=logits.exp()
    probs=counts/counts.sum(1, keepdim=True)
    # loss=-probs[torch.arange(5),ys].log().mean()
    loss=-probs[torch.arange(5),ys].log().mean()+0.01*(W**2).mean()  # 加 L2 正则化防过拟合
    # print(f'loss: {loss.item():.4f}')
    # --- 反向传播 ---
    W.grad=None
    loss.backward()
    # --- 参数更新 ---
    W.data+=-1*W.grad                                  # 学习率 = 1
    LOSS.append(loss.item())
    

plt.plot(range(NUM_ITER), LOSS)                        # 绘制损失曲线
# probs[torch.arange(5),ys].log()
loss

### 从训练好的神经网络采样

训练完成后，我们用学到的权重 $W$ 来生成名字。采样过程和之前相同：从 `'.'` 出发，每次用 Softmax 计算下一个字符的概率，按概率抽样，直到抽到 `'.'` 结束。

注意：这里不再查表 `P[ix]`，而是通过 `one-hot @ W → softmax` 计算概率——同样的结果，但方法可以推广到更复杂的网络。

In [ ]:
# finally,sample from the trained neural network model
g=torch.Generator().manual_seed(2147483647)
for i in range(5):
    out=[]
    ix=0                                               # 从起始符 '.' 开始
    while True:
        #-----
        # before
        # p=P[ix]                                      # 旧方法：直接查概率表

        #-----
        # after                                        # 新方法：通过神经网络计算概率

        xenc=F.one_hot(torch.tensor([ix]), num_classes=27).float()  # one-hot 编码当前字符
        logits=xenc @ W                                # 前向传播
        counts=logits.exp()
        p=counts/counts.sum(1, keepdim=True)           # Softmax 得到概率分布



        
        ix=torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率采样
        
        out.append(itos[ix])
        if ix==0:
            break

    print(''.join(out))